# COMP219
## Lab 7.1: CNN and Attack Strategies

Last week, we introduced MLP, which is the foundational architecture of modern deep learning models. Today, we will have a brief look at convolutional neural networks (CNNs) and we will introduce various attack and defence strategies (you can find the latter in Lab 7.2). The aim of this lab is to prepare you for the assignment by introducing the theory of these strategies and demonstrate them how they work in code. 

### A Brief Introduction to CNNs
CNNs are extremely powerful for processing visual data. They are designed to automatically learn spatial hierarchies of features through backpropagation. These models are primarily used for image recognition, object detection, and image segmentation. The architecture of a CNN model aims to mimic the way biological visual perception works. The model consists of a series of *convolutional layers* which apply filters to the input image to create a *feature map*. These layers help the model learn to identify edges, textures, and patterns. Stacking these layers lets the model learn more and more complex features, which ultimately leads to better performance. 

A CNN model, like any other deep learning model, has an input layer, which accepts data as input and passes it through the network. The convolutional layer performs convolution to the input data, which involves sliding a filter (or kernel) across the input image, performing an element-wise multiplication. The feature map is produced by summing the results of these multiplications.

CNN models also have pooling layers which downsample the feature map produced by the convolutional layer. A common example of this technique is max pooling, which selects the maximum value from a defined region (e.g. 2x2) of the feature map. A typical CNN model will have multiple convolutional and pooling layers fully connected. Each of the layers take the flattened output from the previous layers and the output class scores through an activation function.

#### *Key Terminology*

- __Convolutional layer:__ performs convolution to create a feature map.
- __Kernels:__ a small matrix that focuses on a specific feature of the input (for example, horizontal edges, vertical edges, etc). During the training phase, the network learns the optimal values for the kernels.
- __Feature map:__ The output of the convolutional layer after applying kernels. Each of these maps represents the presence and spatial location of the learned features.
- __Pooling__: a pooling operation is usually applied after producing a feature map. This process reduces the spatial dimensions while keeping the most significant features (reduces compute cost).


In [1]:
import os
import time
import argparse
from typing import Optional, Tuple, Iterable

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms
from torch.optim import LBFGS
from tqdm import tqdm

# -----------------------------
# Args / Config
# -----------------------------
parser = argparse.ArgumentParser(description='MNIST with FGSM / L-BFGS / CW attacks')
parser.add_argument('--batch-size', type=int, default=128)
parser.add_argument('--test-batch-size', type=int, default=128)
parser.add_argument('--epochs', type=int, default=20)
parser.add_argument('--lr', type=float, default=1e-3)
parser.add_argument('--no-cuda', action='store_true', default=False)
parser.add_argument('--seed', type=int, default=42)
parser.add_argument('--model-dir', default='./model-mnist-cnn')
parser.add_argument('--load-model', action='store_true', default=False)
parser.add_argument('--fgsm-eps', type=float, default=0.03)
parser.add_argument('--lbfgs-eps', type=float, default=0.03)
args = parser.parse_args(args=[])

# -----------------------------
# Setup device, seeds
# -----------------------------
use_cuda = not args.no_cuda and torch.cuda.is_available()
device = torch.device('cuda' if use_cuda else 'cpu')
torch.manual_seed(args.seed)
if use_cuda:
    torch.cuda.manual_seed_all(args.seed)

os.makedirs(args.model_dir, exist_ok=True)
if torch.cuda.is_available():
    device = torch.device("cuda")
    print("Using GPU:")
    print(f"  Name: {torch.cuda.get_device_name(device)}")
    print(f"  Total VRAM: {torch.cuda.get_device_properties(device).total_memory / 1024**3:.2f} GB")
    print(f"  Compute Capability: {torch.cuda.get_device_properties(device).major}.{torch.cuda.get_device_properties(device).minor}")
else:
    device = torch.device("cpu")
    print("CUDA not available, using CPU.")

# -----------------------------
# Data
# -----------------------------
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])
trainset = datasets.MNIST('../data', train=True, download=True, transform=transform)
testset = datasets.MNIST('../data', train=False, download=True, transform=transform)
kwargs = {'num_workers': 2, 'pin_memory': True} if use_cuda else {}
train_loader = torch.utils.data.DataLoader(trainset, batch_size=args.batch_size, shuffle=True, **kwargs)
test_loader = torch.utils.data.DataLoader(testset, batch_size=args.test_batch_size, shuffle=False, **kwargs)

Using GPU:
  Name: NVIDIA GeForce RTX 4050 Laptop GPU
  Total VRAM: 6.00 GB
  Compute Capability: 8.9


In [2]:
# -----------------------------
# Convolutional Neural Network Model
# -----------------------------
class Net(nn.Module):
    def __init__(self):
        super().__init__()
        
        # -----------------------------
        # Feature extraction layers
        # -----------------------------
        # These layers learn spatial hierarchies of features (edges, textures, shapes)
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1), #conv layer: input=1 channel (grayscale), output=32 feature maps, kernel=3x3
            nn.BatchNorm2d(32), #BatchNorm: normalise activations to stabilise and speed up training
            nn.ReLU(inplace=True), #ReLU activation: introduces non-linearity
            
            nn.Conv2d(32, 64, 3, padding=1), #conv layer: input=32 channels, output=64 channels, kernel=3x3
            nn.BatchNorm2d(64), #batchNorm
            nn.ReLU(inplace=True), #ReLU
            
            nn.MaxPool2d(2), #Max Pooling: reduces spatial dimensions by 2 (28x28 -> 14x14)
            nn.Dropout(0.25) #dropout: randomly sets 25% of activations to zero (regularisation)
        )
        
        # -----------------------------
        # Fully connected (classification) layers
        # -----------------------------
        self.classifier = nn.Sequential(
            nn.Flatten(), #flatten feature maps to a vector for fully connected layers
            nn.Linear(64 * 14 * 14, 128), #dense layer: maps flattened features to 128 neurons
            nn.ReLU(inplace=True), #ReLU activation
            nn.Dropout(0.5), #dropout: randomly sets 50% of neurons to zero (stronger regularisation)
            nn.Linear(128, 10) #output layer: 10 logits for 10 classes (MNIST digits)
        )
        
        # -----------------------------
        # Weight initialisation
        # -----------------------------
        self._init_weights() #apply proper initialisation to conv and linear layers

    def _init_weights(self):
        #initialise weights to improve training stability
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, nonlinearity='relu') #He/Kaiming initialisation for Conv layers
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0) #Bias initialised to 0
            elif isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight) #Xavier/Glorot initialisation for Linear layers
                nn.init.constant_(m.bias, 0)

    def forward(self, x):
        # Forward pass through the network
        x = self.features(x) #extract features
        x = self.classifier(x) #classify based on features
        return x #return raw logits (CrossEntropyLoss expects logits)


In [3]:
# -----------------------------
# Training / Evaluation Helpers
# -----------------------------

def train_epoch(model, device, loader, optimizer, epoch):
    """
    Train the model for X epochs.
    
    Args:
        model: PyTorch model to train
        device: 'cpu' or 'cuda'
        loader: DataLoader for training data
        optimizer: optimiser for updating model parameters
        epoch: current epoch number
    
    Returns:
        avg_loss: average cross-entropy loss over the epoch
        accuracy: training accuracy over the epoch
    """
    model.train() #set model to training mode (enables dropout, batchnorm updates)
    running_loss = 0.0 #accumulate total loss
    correct = 0 #count correct predictions
    total = 0 #total number of samples processed

    #tqdm loop to show progress bar with live updates
    loop = tqdm(loader, desc=f"Train Epoch {epoch}", leave=False)

    for data, target in loop:
        data, target = data.to(device), target.to(device) #move data and labels to device (CPU or GPU)
        optimizer.zero_grad() #zero gradients from previous step

        outputs = model(data) #forward pass: compute model predictions

        loss = F.cross_entropy(outputs, target) #compute cross-entropy loss
        loss.backward() #backward pass: compute gradients
        
        optimizer.step() #update model weights using optimiser

        #update running totals
        running_loss += loss.item() * data.size(0) #multiply by batch size for correct averaging
        preds = outputs.argmax(dim=1) #predicted class (argmax over logits)
        correct += preds.eq(target).sum().item() #count correct predictions
        total += data.size(0) #accumulate total number of samples

        #update progress bar with current batch loss and cumulative accuracy
        loop.set_postfix(loss=loss.item(), acc=100. * correct / total)

    #return average loss and accuracy over the entire epoch
    return running_loss / total, correct / total


def evaluate(model, device, loader):
    """
    Evaluate the model on a validation or test set.
    
    Args:
        model: PyTorch model to evaluate
        device: 'cpu' or 'cuda'
        loader: DataLoader for evaluation data
    
    Returns:
        avg_loss: average cross-entropy loss over dataset
        accuracy: fraction of correctly predicted samples
    """
    model.eval() #set model to evaluation mode (disables dropout, batchnorm updates)
    loss = 0.0 #accumulate total loss
    correct = 0 #count correct predictions
    total = 0 #total number of samples processed

    #disable gradient computation
    with torch.no_grad():
        for data, target in loader:
            #move data and labels to device (preferably GPU)
            data, target = data.to(device), target.to(device)

            #forward pass
            outputs = model(data)

            #sum up batch loss (reduction='sum') to compute correct average later
            loss += F.cross_entropy(outputs, target, reduction='sum').item()

            #compute predictions
            preds = outputs.argmax(dim=1)

            #count correct predictions
            correct += preds.eq(target).sum().item()

            #  total number of samples
            total += data.size(0)

    #return average loss and accuracy over the dataset
    return loss / total, correct / total


### Attack Strategy: L-BFGS
Limited-memory Broyden-Fletcher-Goldfarb-Shanno (L-BFGS) attack is an optimisation-based method used to generate adversarial examples, which serve as inputs to an ML model. These inputs are designed to mislead the model - the idea is to find small perturbations to input data that will cause the model to make incorrect predictions.

#### Why L-BFGS?
This technique is effective for generating adversarial examples because it is an optimisation-based approach, meaning it tries to find the smallest changes needed to fool the model. Gradient-based methods usually just add noise, but L-BFGS has a closer interaction with the model.

#### How Does It Work?
1. Choose a target input: start with an input (like an image) we want to modify. For example, our model is trained on the MNIST dataset, so let's say we can choose the digit 5. This input label should be classified correctly by the model.
2. Define the loss function: the loss function determines how wrong the model is when predicting the class of the input. It is common practice to use cross-entropy loss, which measures the difference between the true class label and the predicted class probabilities.
3. Set constraints: setting constraints can help us make sure the modified input is similar to the original. An example of this constraint is the L2 norm.
4. Use L-BFGS for optimisation: next, we use the L-BFGS optimisation to find the optimal perturbation that will minimise the loss function while satisfying the constraints. First, we set a small perturbation - could be zeros or random noise. We then calculate the gradient of the loss function with respect to the input. Then, we update the input based on the calculated gradient. We repeat this for a number of iterations.
5. Generate adversarial examples: once the optimisation is complete, we can generate adversarial examples. The modified input should force the model to predict the wrong class, despite being very similar to the original input.

In [4]:
def lbfgs_attack(
    model: nn.Module,
    x: torch.Tensor,
    y: torch.Tensor,
    l2_reg: float = 1e-3,
    max_iter: int = 200,
    lr: float = 0.5,
    epsilon: Optional[float] = None
) -> torch.Tensor:
    """
    Untargeted L-BFGS adversarial attack (per-sample).
    Objective minimszed: -CE(model(x_adv), y) + l2_reg * ||x_adv - x||_2^2

    Args:
        model: classifier
        x: input tensor of shape (1, ...), values assumed in [0,1]
        y: label tensor of shape (1,) (long)
        l2_reg: coefficient for L2 regulariser (smaller -> allow bigger perturbation)
        max_iter: number of LBFGS iterations
        lr: LBFGS learning rate
        epsilon: optional L_inf clipping radius (None = no L_inf clipping)

    Returns:
        x_adv: adversarial example (same shape as x), detached and clamped to [0,1].
    """
    model.eval()
    device = x.device
    dtype = x.dtype

    #keep original copy
    x_orig = x.detach().clone().to(device=device, dtype=dtype)

    #parameter to optimise (LBFGS expects parameters)
    x_adv_param = nn.Parameter(x_orig.clone().detach())

    #LBFGS optimiser
    optimizer = optim.LBFGS([x_adv_param], lr=lr, max_iter=max_iter, line_search_fn='strong_wolfe')

    ce = nn.CrossEntropyLoss(reduction='mean')

    def closure():
        optimizer.zero_grad()
        outputs = model(x_adv_param)
        loss_ce = -ce(outputs, y)  #maximise CE -> minimise -CE
        #per-sample squared L2 then mean
        l2_term = ((x_adv_param - x_orig).view(x_adv_param.size(0), -1).pow(2).sum(dim=1)).mean()
        loss = loss_ce + l2_reg * l2_term
        #backward
        loss.backward()
        return loss

    try:
        optimizer.step(closure)
    except Exception as e:
        #LBFGS may sometimes throw numerical errors
        pass

    #get optimised tensor, ensure on proper device/dtype and clamp
    x_adv = x_adv_param.detach().to(device=device, dtype=dtype)
    if epsilon is not None:
        perturbation = torch.clamp(x_adv - x_orig, -epsilon, epsilon)
        x_adv = torch.clamp(x_orig + perturbation, 0.0, 1.0).detach()
    else:
        x_adv = torch.clamp(x_adv, 0.0, 1.0).detach()

    return x_adv


def evaluate_lbfgs(
    model: nn.Module,
    loader,
    device: torch.device,
    epsilon: Optional[float] = None,
    l2_reg: float = 1e-3,
    max_iter: int = 100,
    lr: float = 0.5,
    num_restarts: int = 3
) -> float:
    """
    Evaluate model accuracy under per-sample L-BFGS attack with optional random restarts.

    Args:
        model: classifier
        loader: dataloader yielding (data, target)
        device: torch.device
        epsilon: optional L_inf clipping radius for lbfgs_attack
        l2_reg: L2 regulariser passed to lbfgs_attack
        max_iter: LBFGS max_iter per sample
        lr: LBFGS learning rate
        num_restarts: number of random restarts per sample (higher = stronger attack)

    Returns:
        accuracy on adversarially perturbed dataset (float in [0,1])
    """
    model.to(device)
    model.eval()
    correct = 0
    total = 0

    for data, target in tqdm(loader, desc='L-BFGS Eval', leave=False):
        data, target = data.to(device), target.to(device)
        B = data.size(0)
        adv_batch = torch.zeros_like(data)

        for i in range(B):
            xi = data[i:i+1].to(device)
            yi = target[i:i+1].to(device)
            best_adv = None
            best_loss = float('-inf')

            for r in range(num_restarts):
                # optional small random noise for restarts
                if r > 0:
                    xi_restart = xi + 0.01 * torch.randn_like(xi)
                    xi_restart = torch.clamp(xi_restart, 0.0, 1.0)
                else:
                    xi_restart = xi.clone()

                adv_candidate = lbfgs_attack(
                    model,
                    xi_restart,
                    yi,
                    l2_reg=l2_reg,
                    max_iter=max_iter,
                    lr=lr,
                    epsilon=epsilon
                )

                # compute current negative CE (attack objective)
                with torch.no_grad():
                    outputs = model(adv_candidate)
                    ce_loss = nn.CrossEntropyLoss(reduction='none')(outputs, yi)
                    objective = ce_loss.item()  # negative CE is already handled inside lbfgs

                # select adversarial candidate with highest loss
                if objective > best_loss:
                    best_loss = objective
                    best_adv = adv_candidate

            adv_batch[i:i+1] = best_adv

        # evaluate on adversarial batch
        with torch.no_grad():
            out = model(adv_batch)
            pred = out.argmax(dim=1)
            correct += (pred == target).sum().item()
            total += target.size(0)

    return correct / total if total > 0 else 0.0

### Attack Strategy: FGSM
Fast gradient sign method (FGSM) is a simple but powerful attack technique that generates adversarial examples by adding small peturbations to input data. Just like L-BFGS, FGSM aims to force a model to misclassify the inputs.

#### Why FGSM?
Unlike L-BFGS, FGSM relies on the gradients of the model to determine how to adjust the input in a single step. This makes it suitable for quick adversarial attacks. FGSM is very useful when we need to evaluate a model's robustness to adversarial attacks quickly.

#### How Does It Work?
1. Choose target input: start with an input that the model is trained to classify correctly. For example, the digit 5 from the MNIST dataset.
2. Define the loss function: for classification tasks, we can use cross-entropy loss.
3. Compute gradients: calculate the gradient of the loss function with respect to the input image. The gradient indicates how to change each pixel of the input to increase the loss, which makes the model more likely to misclassify the image.
4. Generate perturbation: the perturbation is created by taking the sign of the gradient. The sign function keeps only the direction of the change (positive or negative) for each pixel. This signed gradient is then multiplied by a small value (epsilon) which controls the intensity of the perturbation. Smaller epsilon value leads to less visible changes (which may be less effective in causing misclassification).
5. Apply perturbation: Add the perturbation to the original input to create the adversarial example.
6. Clamp the values: after adding the perturbation, ensure the resulting adversarial example remains within valid pixel ranges. Values outside the valid range cannot be processed by the model.

In [5]:
def fgsm_attack(model: nn.Module, x: torch.Tensor, y: torch.Tensor, epsilon: float = 0.031) -> torch.Tensor:
    """
    Perform an untargeted FGSM attack on a batch of inputs x (shape [N, ...]) with labels y.

    Args:
        model: torch.nn.Module, the classifier (should be on the same device as x).
        x: input tensor, values assumed in [0, 1] (or in the model's expected range).
        y: target labels (long tensor).
        epsilon: L_inf perturbation size (per-feature max change).

    Returns:
        x_adv: adversarial examples (tensor detached from autograd), clipped to [0,1].
    """
    
    x_adv = x.clone().detach().requires_grad_(True) #make a fresh copy of the input and tell autograd to compute gradients wrt it

    model.eval() #model to be in eval mode for deterministic behaviour (dropout/batchnorm)

    #clear any existing gradients on the model (not strictly necessary when only computing input grad,
    #but good hygiene if the model was used for training earlier)
    model.zero_grad()

    outputs = model(x_adv) #forward pass
    
    loss = nn.CrossEntropyLoss()(outputs, y) #compute loss, default reduction='mean' 
    loss.backward() #backprop

    #create perturbation by taking the sign of the input gradient (elementwise) and scaling by epsilon
    #x_adv.grad is the gradient dL/dx for each input element
    perturbation = epsilon * x_adv.grad.data.sign()

    #apply the perturbation and clamp to the valid input range.
    x_adv = x_adv + perturbation
    x_adv = torch.clamp(x_adv, 0.0, 1.0).detach()  #detach so returned tensor does not keep graph

    return x_adv


def evaluate_fgsm(model: nn.Module, data_loader: Iterable, epsilon: float, device: torch.device) -> float:
    """
    Generate adversarial examples using FGSM for a dataset and evaluate model accuracy on them.

    Args:
        model: classifier (should be moved to `device` externally or within this function).
        data_loader: iterable of (data, target) batches.
        epsilon: FGSM step size.
        device: torch.device where tensors should be located.

    Returns:
        adv_accuracy: accuracy on adversarially perturbed dataset (float in [0,1]).
    """
    model.to(device)
    model.eval()

    correct_adv = 0
    total = 0

    for data, target in data_loader:
        #move batch to device
        data, target = data.to(device), target.to(device)

        #create adversarial examples (this computes gradients w.r.t. inputs)
        adv_data = fgsm_attack(model, data, target, epsilon)

        #evaluate model on adversarial examples without tracking gradients
        with torch.no_grad():
            output = model(adv_data)
            pred = output.argmax(dim=1)
            correct_adv += (pred == target).sum().item()
            total += target.size(0)

    adv_accuracy = correct_adv / total if total > 0 else 0.0
    return adv_accuracy


### Carlini-Wagner Attack
The CW attack is an optimisation-based attack that aims to find small changes to the input to make the model misclassify it (similar to L-BFGS). The motivation of the CW attack is to produce high quality adversarial examples with minimal distortion. Unlie L-BFGS, it uses a margin-style objective and a tanh reparameterisation to push the model logits toward a wrong class (or a target class)while keeping the perturbation extremely small. Practically, the CW attack produces smaller, higher quality L2 perturbations than L-BFGS.

#### Why CW?
If you care about minimising your L2 perturbations (e.g. creating more realistic adversarial examples), this attack might suit you better than any of the previous examples. It is also stronger than FGSM because it is iterative (while FGSM is one-shot) and we also have the luxury of introducing a box constraint via the tanh/arctanh reparameterisation which results pixel staying within bounds and the perturbed image will look more natural.

#### How Does It Work?
1. Start with a correctly classified input
2. Define an ojbective that forces the model to misclassify the input while keeping the input more natural
3. Adjust the input using gradients from the model (iteratively)
4. Track the best adversarial version that successfully fools the model
5. Generate the adversarial example.

In [6]:
# Carlini–Wagner L2 attack pseudocode
# Purpose: find a tiny L2 perturbation that makes the model misclassify.

function CW_L2_Attack(model, x_orig, y_true, c=1e-1, kappa=0, max_iters=1000, lr=1e-2):
    #x_orig: single input in pixel space [0,1] (or denormalise/renormalise consistently - see previous code cells)
    #model: returns logits for input (model must be in eval mode)
    #c: trade-off constant between distortion and classification objective
        #how strongly the attack pushes the model logit margin vs how small the perturbation stays
        #hint: try several c values
    #kappa: confidence margin (>=0 makes adversarial more confident)
    #returns: x_adv (same shape as x_orig)

    # 1) reparameterise so we can enforce pixel bounds easily:
    #use w such that x_adv = 0.5 * (tanh(w) + 1)  -> x_adv always in [0,1].
    #w = arctanh(2 * clamp(x_orig, eps, 1-eps) - 1)   (invert tanh safely, purpose is to keep pixels valid without clamping = more stable optimisation)

    #2) optimiser over w (e.g., Adam)
    optimizer = Adam([w], lr=lr)

    best_adv = x_orig
    best_dist = +inf

    for iter in 1..max_iters:
        optimizer.zero_grad()

        #3)map w -> candidate adversarial in pixel space
        x_candidate = 0.5 * (tanh(w) + 1) #guaranteed in [0,1]

        #4) evaluate model on the *normalized* input the model expects
        logits = model(preprocess_for_model(x_candidate))

        #5) compute margin-based classification term (f)
        #targeted: push target logit above others
        #untargeted: push some other logit above true logit
        f = margin_loss_from_logits(logits, y_true, target, kappa)

        #6) compute L2 distortion (in pixel space) and total loss
        l2 = L2_distance(x_candidate, x_orig)
        loss = l2 + c * f

        #7) backprop and step
        loss.backward()
        optimizer.step()

        #8) track best successful adversarial (smallest L2 that made model wrong)
        if success_condition(logits, y_true, target, kappa):
            if l2 < best_dist:
                best_dist = l2
                best_adv = x_candidate.detach()

    #9) return best found (or final candidate)
    return best_adv


SyntaxError: invalid syntax (4233692674.py, line 4)

In [7]:
# -----------------------------
# Evaluation wrappers
# -----------------------------

def evaluate_fgsm(model, loader, device, epsilon):
    model.eval()
    correct = 0
    total = 0
    for data, target in tqdm(loader, desc='FGSM Eval', leave=False):
        data, target = data.to(device), target.to(device)
        adv = fgsm_attack(model, data, target, epsilon)
        out = model(adv)
        pred = out.argmax(dim=1)
        correct += (pred == target).sum().item()
        total += target.size(0)
    return correct / total


def evaluate_lbfgs(model, loader, device, epsilon=None, l2_reg=1e-2, max_iter=20):
    model.eval()
    correct = 0
    total = 0
    for data, target in tqdm(loader, desc='L-BFGS Eval', leave=False):
        data, target = data.to(device), target.to(device)
        B = data.size(0)
        adv_batch = torch.zeros_like(data)
        for i in range(B):
            xi = data[i:i+1]
            yi = target[i:i+1]
            adv = lbfgs_attack(model, xi, yi, l2_reg=l2_reg, max_iter=max_iter, epsilon=epsilon)
            adv_batch[i:i+1] = adv
        out = model(adv_batch.to(device))
        pred = out.argmax(dim=1)
        correct += (pred == target).sum().item()
        total += target.size(0)
    return correct / total

In [8]:
# -----------------------------
# Main
# -----------------------------

def main():
    model = Net().to(device)
    optimizer = optim.Adam(model.parameters(), lr=args.lr)

    if args.load_model and os.path.exists(os.path.join(args.model_dir, 'final_model.pt')):
        model.load_state_dict(torch.load(os.path.join(args.model_dir, 'final_model.pt'), map_location=device))
        print('Loaded model')
    else:
        best_acc = 0.0
        for epoch in range(1, args.epochs + 1):
            start = time.time()
            trn_loss, trn_acc = train_epoch(model, device, train_loader, optimizer, epoch)
            tst_loss, tst_acc = evaluate(model, device, test_loader)
            print(f"Epoch {epoch:02d} | Train Loss: {trn_loss:.4f}, Train Acc: {trn_acc*100:.2f}% | Test Loss: {tst_loss:.4f}, Test Acc: {tst_acc*100:.2f}% | Time: {time.time()-start:.1f}s")
            if tst_acc > best_acc:
                best_acc = tst_acc
                torch.save(model.state_dict(), os.path.join(args.model_dir, 'final_model.pt'))
        print(f"Best Test Acc: {best_acc*100:.2f}%")

    # Load best model
    model.load_state_dict(torch.load(os.path.join(args.model_dir, 'final_model.pt'), map_location=device))

    #evaluate clean
    clean_acc = evaluate(model, device, test_loader)
    print(f"Clean test acc: {clean_acc[1]*100:.2f}%")

    #FGSM
    fgsm_acc = evaluate_fgsm(model, test_loader, device, epsilon=args.fgsm_eps)
    print(f"FGSM (eps={args.fgsm_eps}) acc: {fgsm_acc*100:.2f}%")

    #L-BFGS (per-sample)
    lbfgs_acc = evaluate_lbfgs(model, test_loader, device, epsilon=args.lbfgs_eps, l2_reg=1e-2, max_iter=20)
    print(f"L-BFGS (eps={args.lbfgs_eps}) acc: {lbfgs_acc*100:.2f}%")

if __name__ == '__main__':
    main()


Epoch 01 | Train Loss: 0.6225, Train Acc: 80.01% | Test Loss: 0.0814, Test Acc: 97.66% | Time: 21.6s


Epoch 02 | Train Loss: 0.3049, Train Acc: 89.71% | Test Loss: 0.0668, Test Acc: 97.89% | Time: 31.3s


Epoch 03 | Train Loss: 0.2555, Train Acc: 91.44% | Test Loss: 0.0565, Test Acc: 98.20% | Time: 30.8s


Epoch 04 | Train Loss: 0.2233, Train Acc: 92.46% | Test Loss: 0.0492, Test Acc: 98.52% | Time: 30.0s


Epoch 05 | Train Loss: 0.2035, Train Acc: 93.03% | Test Loss: 0.0452, Test Acc: 98.58% | Time: 30.9s


...

JSMA Eval:  96%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████      | 76/79 [1:51:52<04:22, 87.52s/it]

JSMA Eval:  97%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████    | 77/79 [1:53:20<02:55, 87.64s/it]

JSMA Eval:  99%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████  | 78/79 [1:54:45<01:26, 86.95s/it]

JSMA Eval: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 79/79 [1:54:57<00:00, 64.29s/it]

JSMA (theta=0.1, gamma=0.1) acc: 98.26%
